In [ ]:
#This code is responsible for extracting entso and smard data using an api key"


import requests
import xml.etree.ElementTree as ET
import pandas as pd
from datetime import datetime
from dateutil.relativedelta import relativedelta
from google.colab import drive
drive.mount('/content/drive')


API_KEY = "[REDACTED]"
BASE_URL = "https://web-api.tp.entsoe.eu/api"
DOMAIN = "10Y1001A1001A82H"

def download_prices(start_str, end_str, market_type="dayahead"):
    start = datetime.strptime(start_str, "%Y%m%d%H%M")
    end = datetime.strptime(end_str, "%Y%m%d%H%M")

    all_rows = []
    chunk_start = start

    while chunk_start < end:
        chunk_end = min(chunk_start + relativedelta(months=1), end)

        params = {
            "securityToken": API_KEY,
            "documentType": "A44",
            "in_Domain": DOMAIN,
            "out_Domain": DOMAIN,
            "periodStart": chunk_start.strftime("%Y%m%d%H%M"),
            "periodEnd": chunk_end.strftime("%Y%m%d%H%M"),
        }

        if market_type == "intraday_A05":
            params["contract_MarketAgreement.Type"] = "A05"
        elif market_type == "intraday_A07":
            params["contract_MarketAgreement.Type"] = "A07"


        r = requests.get(BASE_URL, params=params)

        if r.status_code != 200:
            print(f"  HTTP Error {r.status_code} for {chunk_start.strftime('%Y-%m')}")
            print(f"  Response: {r.text[:300]}")
            chunk_start = chunk_end
            continue

        root = ET.fromstring(r.content)
        ns = {"ns": root.tag.split("}")[0].strip("{")}

        reason = root.find(".//ns:Reason/ns:text", ns)
        if reason is not None:
            print(f"  API message for {chunk_start.strftime('%Y-%m')}: {reason.text}")
            chunk_start = chunk_end
            continue

        chunk_rows = 0
        for ts in root.findall(".//ns:TimeSeries", ns):
            for period in ts.findall(".//ns:Period", ns):
                start_node = period.find(".//ns:timeInterval/ns:start", ns)
                if start_node is None:
                    continue

                period_start = datetime.fromisoformat(
                    start_node.text.replace("Z", "+00:00")
                )

                resolution = period.find("ns:resolution", ns).text
                if resolution == "PT60M":
                    step = pd.Timedelta(hours=1)
                elif resolution == "PT15M":
                    step = pd.Timedelta(minutes=15)
                else:
                    step = pd.Timedelta(hours=1)

                for point in period.findall("ns:Point", ns):
                    position = int(point.find("ns:position", ns).text)
                    price_node = point.find("ns:price.amount", ns)
                    if price_node is None:
                        continue
                    price = float(price_node.text)
                    timestamp = period_start + (position - 1) * step
                    all_rows.append([timestamp, price])
                    chunk_rows += 1

        print(f"  ✓ {chunk_start.strftime('%Y-%m')} — {chunk_rows} new rows — {len(all_rows)} total")
        chunk_start = chunk_end

    if not all_rows:
        print(f"  WARNING: No data returned for market_type={market_type}")
        return pd.DataFrame(columns=["timestamp", "price_eur_mwh"])

    df = pd.DataFrame(all_rows, columns=["timestamp", "price_eur_mwh"])
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    df = df.sort_values("timestamp").drop_duplicates("timestamp").reset_index(drop=True)
    return df


def filter_trading_window(df):
    """Keep only 11:30-23:30 UTC"""
    if df.empty:
        return df
    return df[df["timestamp"].dt.hour.between(11, 23)].reset_index(drop=True)


# -------------------------
# Winter seasons only
# -------------------------
winter_ranges = [
    ("202210010000", "202302280000"),   # Winter 22/23  → training
    ("202310010000", "202402290000"),   # Winter 23/24  → training
    ("202410010000", "202502280000"),   # Winter 24/25  → test set
]

# -------------------------
# Day-ahead prices
# -------------------------
print("=" * 50)
print("DOWNLOADING DAY-AHEAD PRICES")
print("=" * 50)

dayahead_frames = []
for start, end in winter_ranges:
    print(f"\n--- Day-ahead {start[:6]} to {end[:6]} ---")
    df = download_prices(start, end, market_type="dayahead")
    df["season"] = f"{start[:4]}/{str(int(start[:4])+1)[2:]}"
    dayahead_frames.append(df)

dayahead = pd.concat(dayahead_frames).reset_index(drop=True)
dayahead = filter_trading_window(dayahead)
dayahead["market"] = "dayahead"



# -------------------------
# Summary
# -------------------------
print("\n" + "=" * 50)
print("SUMMARY")
print("=" * 50)
print(f"Day-ahead rows:  {len(dayahead)}")


print("\nDay-ahead by season:")
if not dayahead.empty:
    print(dayahead.groupby("season")["price_eur_mwh"].describe().round(2))


print("\nDay-ahead sample:")
print(dayahead.head())

# -------------------------
# Save to CSV
# -------------------------
dayahead.to_csv("/content/drive/MyDrive/MEng Project/dayahead_prices.csv", index=False)

